In [1]:
%load_ext autoreload
%autoreload 2

# 1. Environment

In [1]:
from dotenv import load_dotenv
from openai import OpenAI
import os

load_dotenv()

openrouter_client = OpenAI(
    api_key=os.environ.get("OPENROUTER_API_KEY"),
    base_url="https://openrouter.ai/api/v1",
)


# 2. RAG

In [2]:
def llm(prompt):
    response = openrouter_client.responses.create(
        model="openrouter/owl-alpha",
        input=prompt,
    )
    return response.output_text

question = 'I just discovered the course. Can I join now?'
answer = llm(question)
print(answer)

Hello there! I'm OWL. Yes, you're welcome to join the course now! It's exciting to have you here. Let's get you started on your learning journey with us. Is there anything specific you'd like to know about the course or how to get started?


In [ ]:
from pprint import pprint
pprint(answer)

("Of course! I'd be happy to help you with information about joining the "
 "course. Could you please provide more details about which course you're "
 'referring to? That way, I can give you the most accurate and helpful '
 'response.')


Adding context manually

In [3]:
context = '''
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we're still accepting submissions.

Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

What is the video/zoom link to the stream for the "Office Hours" or live/workshop sessions?
The zoom link is only published to instructors/presenters/TAs. Students participate via YouTube Live and submit questions to Slido.

Cloud alternatives with GPU
Check the quota and reset cycle carefully. Potential options include Google Colab, Kaggle, Databricks.
'''
prompt = f'''
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."

Question:
{question}

Context:
{context}
'''

In [4]:
# answer = llm(prompt)
print(answer)

Hello there! I'm OWL. Yes, you're welcome to join the course now! It's exciting to have you here. Let's get you started on your learning journey with us. Is there anything specific you'd like to know about the course or how to get started?


# 3. Dataset

In [5]:
import requests

docs_url = 'https://datatalks.club/faq/json/courses.json'
response = requests.get(docs_url)
courses_raw = response.json()

In [6]:
documents = []
url_prefix = 'https://datatalks.club/faq'

for course in courses_raw:
    course_url = f'{url_prefix}{course['path']}'

    course_response = requests.get(course_url)
    course_response.raise_for_status()
    course_data = course_response.json()

    documents.extend(course_data)

len(documents)

1208

In [7]:
documents[0]

{'id': '0e38656cfb',
 'course': 'machine-learning-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'How do I submit homework?',
 'answer': "- Do the tasks locally\n- Publish your code (e.g., in your own GitHub repo)\n- Submit your answers via the homework form and include the URL to your code\n- You will see the answers only after the deadline\n- Homeworks are in the cohorts folder, e.g. for 2025 it's [`cohorts/2025`](https://github.com/DataTalksClub/machine-learning-zoomcamp/tree/master/cohorts/2025)\n- The forms for submitting the homework are in the [course management platform](https://courses.datatalks.club/)"}

# 4. Search
Indexing with minsearch

In [8]:
from minsearch import Index

index = Index(
    text_fields=['question', 'section', 'answer'],
    keyword_fields=['course']
)

index.fit(documents)

In [9]:
question = 'I just discovered the course. Can I join now?'

search_results = index.search(
    question,
    boost_dict={'question': 2., 'section': 0.5},
    filter_dict={'course': 'llm-zoomcamp'},
    num_results=5
)

search_results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you c

In [10]:
[doc['question'] for doc in search_results]

['I just discovered the course. Can I still join?',
 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
 'When will the course be offered next?',
 'I missed the first homework - can I still get a certificate?']

Wrapping it in a function

In [11]:
def search(question, course='llm-zoomcamp'):
    boost_dict = {'question': 2.0, 'section': 0.5}
    filter_dict = {'course': course}

    return index.search(
        question,
        boost_dict=boost_dict,
        filter_dict=filter_dict,
        num_results=5
    )

search_results = search(question)
search_results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you c

# 5. Building the Prompt

In [12]:
INSTRUCTIONS = '''
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
'''
USER_PROMPT_TEMPLATE = '''
Question:
{question}

Context:
{context}
'''

In [13]:
def build_context(search_results):
    lines = []

    for doc in search_results:
        lines.append(doc['section'])
        lines.append('Q: ' + doc['question'])
        lines.append('A: ' + doc['answer'])
        lines.append('')

    return '\n'.join(lines).strip()


def build_prompt(question, search_results):
    context = build_context(search_results)
    prompt = USER_PROMPT_TEMPLATE.format(
        question=question,
        context=context
    )
    return prompt.strip()

In [14]:
prompt = build_prompt(question, search_results)

print(prompt)

Question:
I just discovered the course. Can I join now?

Context:
General Course-Related Questions
Q: I just discovered the course. Can I still join?
A: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

General Course-Related Questions
Q: Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
A: You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

General Course-Related Questions
Q: Certificate: Can I follow the course in a self-paced mode and get a certificate?
A: No, you can only get a certificate if you finish the course with a "live" cohort.

We don't award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project

# 6. The LLM


In [15]:
model='openrouter/owl-alpha'

response = openrouter_client.responses.create(
    model=model,
    input=prompt
)

In [16]:
response.output[0]


ResponseOutputMessage(id='msg_tmp_1n31z72yulv', content=[ResponseOutputText(annotations=[], text='Yes, you can still join the course! However, if you want to receive a certificate, you’ll need to submit your project while we’re still accepting submissions.', type='output_text', logprobs=[])], role='assistant', status='completed', type='message')

In [17]:
# That's a lot of digging
response.output[0].content[0].text


'Yes, you can still join the course! However, if you want to receive a certificate, you’ll need to submit your project while we’re still accepting submissions.'

In [18]:
# There's a shortcut:
response.output_text

'Yes, you can still join the course! However, if you want to receive a certificate, you’ll need to submit your project while we’re still accepting submissions.'

In [19]:
response.usage

ResponseUsage(input_tokens=434, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=36, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=470, cost=0, is_byok=False, cost_details={'upstream_inference_cost': 0, 'upstream_inference_input_cost': 0, 'upstream_inference_output_cost': 0})

Calculating the price

In [20]:
input_price = 0.75 / 1_000_000
output_price = 4.50 / 1_000_000

cost = (
    response.usage.input_tokens * input_price +
    response.usage.output_tokens * output_price
)

cost

0.0004875

In [21]:
message_history = [
    {'role': 'developer', 'content': INSTRUCTIONS},
    {'role': 'user', 'content': prompt}
]

response = openrouter_client.responses.create(
    model=model,
    input=message_history
)

The LLM function

In [22]:
def llm(instructions, user_prompt, model=model):
    message_history = [
        {'role': 'developer', 'content': instructions},
        {'role': 'user', 'content': user_prompt}
    ]

    response = openrouter_client.responses.create(
        model=model,
        input=message_history
    )

    return response.output_text

Full RAG

In [23]:
def rag(query, model='openrouter/owl-alpha'):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(INSTRUCTIONS, prompt, model=model)
    return answer

answer = rag('I just discovered the course. Can I join now?')
print(answer)

Yes, you can still join the course. However, if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.


In [24]:
answer = rag('How do I get a certificate?')
print(answer)

To get a certificate, you must complete the course with a "live" cohort. Certificates are not awarded for the self-paced mode because you need to peer-review 3 capstone projects after submitting your own, and peer-reviewing is only possible while the course is running. Additionally, you need to pass the Capstone project to get the certificate. Homework is not mandatory, but it is recommended for reinforcing concepts and the points count towards your rank on the leaderboard.


# 7. rag-helper

In [39]:
from dotenv import load_dotenv
from src import FaqHttpLoader, MinsearchIndex, RAGBase, OpenRouterClient

load_dotenv()

loader = FaqHttpLoader()
documents = loader.load()
print(f"Loaded {len(documents)} documents")

Loaded 1208 documents


In [ ]:
index = MinsearchIndex(documents)

assistant = RAGBase(
    index=index,
    llm_client=OpenRouterClient(),
    llm_model="openrouter/owl-alpha",
    course_filter="data-engineering-zoomcamp"
    )

answer = assistant.rag('I just discovered the course. Can I join now?')
print(answer)

Yes, you can still join the course even after the start date. You are eligible to submit homework, but be aware of deadlines for assignments and the final project, so it's best not to leave everything to the last minute.


In [38]:
print(assistant.rag('How do I get a certificate?'))
print(assistant.rag('Can I still join the course after it started?'))

To get your certificate, follow these steps:

1. **Wait for two announcements** in the course Telegram and Slack channels:
   - The first announcement will ask you to **verify your name** is correct on the certificate. If it's not, edit it in your course profile under "Edit Course Profile."
   - The second announcement will confirm that **grading is complete** and the certificate is ready.

2. After the second announcement, **log into your enrollment page** on the cohort's course site:
   - URL format: `https://courses.datatalks.club/de-zoomcamp-<year>/enrollment`
   - The current cohort's URL is available in the [course repo](https://github.com/DataTalksClub/data-engineering-zoomcamp).

3. Follow the instructions in [certificates.md](https://github.com/DataTalksClub/data-engineering-zoomcamp/blob/main/certificates.md) to generate your certificate document.

**Important Notes:**
- You **do not need to complete homeworks** to get the certificate, as long as you complete the peer-reviewe

# 8. data-ingestion

In [ ]:
loader = FaqHttpLoader()
documents = loader.load()

print(f"Loaded {len(documents)} documents")

Loaded 1208 documents


In [41]:
docs_llm = [doc for doc in documents if doc['course'] == 'llm-zoomcamp']
print(f'LLM Zoomcamp: {len(docs_llm)} documents')

LLM Zoomcamp: 79 documents


In [ ]:
import time
from sqlitesearch import TextSearchIndex
from tqdm.auto import tqdm

index = TextSearchIndex(
    text_fields=['question', 'section', 'answer'],
    keyword_fields=['course'],
    db_path='faq.db'
)

# instead of adding documents one by one
# we can just use index.fit(documents) at the beginning
# and then add new documends if needed later

for doc in tqdm(docs_llm):
    index.add(doc)

index.close()
print('Done. Index saved to faq.db')

  0%|          | 0/79 [00:00<?, ?it/s]

Done. Index saved to faq.db


Connect to the same database:

In [57]:
from sqlitesearch import TextSearchIndex

sqlite_index = TextSearchIndex(
    text_fields=['question', 'section', 'answer'],
    keyword_fields=['course'],
    db_path='faq.db'
)
sqlite_index.count()

316

Try a search:

In [58]:
results = sqlite_index.search('Can I still join the course after it started?', num_results=5)
[doc['question'] for doc in results]

['I just discovered the course. Can I still join?',
 'I just discovered the course. Can I still join?',
 'I just discovered the course. Can I still join?',
 'I just discovered the course. Can I still join?',
 'I missed the first homework - can I still get a certificate?']

RAG with sqlitesearch

In [62]:
from src import RAGBase, OpenRouterClient

# from openai import OpenAI

# openai_client = OpenAI()

# assistant = RAGBase(
#     index=sqlite_index,
#     llm_client=openai_client,
# )
assistant = RAGBase(
    index=sqlite_index,
    llm_client=OpenRouterClient(),
    llm_model="openrouter/owl-alpha",
    course_filter="llm-zoomcamp"
    )

answer = assistant.rag('I just discovered the course. Can I join now?')
print(answer)

Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.


In [63]:
answer = assistant.rag('Can I still join the course after it started?')
print(answer)

Yes, you can still join the course after it has started. However, if you want to receive a certificate, you need to submit your project while submissions are still being accepted.
